<a href="https://colab.research.google.com/github/Snowmass22/AIML/blob/main/dog_cat_cnn_classfication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("salader/dogsvscats")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dogsvscats' dataset.
Path to dataset files: /kaggle/input/dogsvscats


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [3]:
import os

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# Using the actual dataset path dynamically
train_dir = os.path.join(path, 'train')
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


In [4]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.convo_layers=nn.Sequential(
        nn.Conv2d(3,16,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(16,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        # Fixed: Input channels changed from 16 to 32 to match previous layer's output
        nn.Conv2d(32,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )
    self.fc=nn.Sequential(
        # Fixed: Since 224x224 input yields 28x28 after three MaxPool2d(2,2) operations
        nn.Linear(32*28*28,500),
        nn.ReLU(),
        nn.Linear(500,2)
    )
  def forward(self,x):
    x=self.convo_layers(x)
    x=x.view(x.size(0),-1)
    # Fixed: Changed self.fc_layers to self.fc
    x=self.fc(x)
    return x

In [5]:
model=CNN()

In [6]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [ ]:
#training the model
num_epochs=10
for epoch in range(num_epochs):
  epoch_training_loss=0.0

  for images,labels in train_loader:
    optimizer.zero_grad()
    # Fixed: Use model(images) instead of model.forward(images)
    output=model(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss+=loss.item()
  print(f"Epoch [{epoch+1}/{num_epochs}], training loss = {epoch_training_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
correct=0
total=0
with torch.no_grads():
  for images,label in test_loader:
    outputs=model.forward(images)
    _,predicated=torch.max(outputs,1)

    correct+=(predicated==labels)
    total+=labels.size(0)
print(f"accuracy={correct/total*100}")